# Lesson 11 — LLMs: First API Call

**The bridge from Lesson 6.** Lesson 6 made your first API call over HTTP: a
request to a URL, a JSON body, a few headers, a JSON response you turned into
a table. Everything worked because the endpoint kept a quiet promise — the same
request shape every time, and values that are *true by construction* (a weather
API's temperature is the measured temperature).

Today we call one more API. The request looks almost identical to Lesson 6's.
But the thing on the other end is a **large language model**, and its reply is
the first data in this whole course with **no guaranteed properties at all** —
not its shape, not its type, not its truth.

```
A question in plain English        "Which category is this headline?"
        ↓
An API call            (Unit 2)     same request shape as Lesson 6
        ↓
Text comes back        (Unit 1)     plausible — and nothing more
        ↓
Give it a shape        (Unit 3)     ask for JSON, validate every reply
        ↓
Give it trust          (Unit 4)     verify numbers in Python, mind privacy + cost
```

The through-line of the whole course carries straight through: a structure is
defined by its **properties**. Lesson 1 gave values their types; Lesson 2 asked
which properties a file format keeps or loses; Lesson 6 got them from a
well-behaved API. An LLM gives you none for free. Every property you want —
"this is JSON", "the category is one of six", "the number is right" — you assign
yourself, in Python, after the reply lands.

**Prerequisites and setup**

- Lessons 1–6: types and properties, pandas, validate-early, functions, and the REST call.
- Uses `pandas`, `requests`, and (optionally) the `anthropic` SDK — the same course environment.
- **Runs fully offline.** With no API key set, every call replays a recorded
  response, so nothing here needs the network or spends a cent.
- Run top to bottom — later cells depend on earlier ones.

**Suggested sessions:** ① Units 1–2 · ② Units 3–4

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

print("pandas", pd.__version__)

# The anthropic SDK ships in this environment; we only USE it if a real API
# key is set (it won't be, in class).
try:
    import anthropic
    print("anthropic", anthropic.__version__)
except ImportError:
    anthropic = None
    print("anthropic SDK not installed — fine, we run offline")

# Runtime artifacts live under ../data/ (gitignored), never in the repo tree.
FIXTURE_DIR = Path("../data/lesson11/fixtures")
FIXTURE_DIR.mkdir(parents=True, exist_ok=True)

# Is a real key set? Almost certainly not — and the whole lesson is built to
# run without one by replaying recorded responses.
HAVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("ANTHROPIC_API_KEY set?", HAVE_KEY, "->",
      "could call live" if HAVE_KEY else "will replay fixtures")

## Unit 1 — What an LLM is  ·  ~35 min

A large language model is, mechanically, one repeated move — Lesson 4 would call
it an *algorithm*: given a run of text, predict the next chunk. Append it, then
predict again. That's the whole engine. Answering questions, writing code,
classifying headlines — all of it is that one move, run over and over.

Four words make the rest of the lesson make sense: **tokens**, **next-token
prediction**, **context window**, **temperature**.

### Tokens — what the model actually reads

A model doesn't see characters or words. It sees **tokens**: subword chunks
(` inflation`, `ation`, ` 3`, `.9`). A rough rule of thumb for English is
~4 characters per token, but the exact split comes from the model's *tokenizer*,
not from us. We can't run the real tokenizer offline, so we estimate — and stay
honest that it's an estimate.

In [ ]:
def estimate_tokens(text):
    """Rough English rule of thumb: ~4 characters per token. The AUTHORITATIVE
    count comes from the model's tokenizer, or the API's usage field — this is
    only for budgeting."""
    return max(1, round(len(text) / 4))

sample = "Unemployment falls to 3.9% as hiring accelerates"
print(repr(sample))
print("characters:", len(sample), "| estimated tokens:", estimate_tokens(sample))

### Next-token prediction and temperature

To make "predict the next token" concrete, here is a toy imitation: a model
built from a handful of sentences that, given the last word, picks a likely next
one. A real model weighs thousands of tokens of context and was trained on
trillions — but the *move* is the same, and this toy shows two things at once:
how text is generated, and what **temperature** does to it.

In [ ]:
import random
from collections import defaultdict, Counter

corpus = """
inflation cooled again as consumer prices rose modestly
consumer prices rose again as inflation stayed high
the central bank held interest rates steady
the central bank raised interest rates again
markets rose as the central bank held rates steady
""".lower().split()

# Count which word tends to follow which (a "bigram" model).
following = defaultdict(Counter)
for first, second in zip(corpus, corpus[1:]):
    following[first][second] += 1

def next_word(word, temperature, rng):
    choices = following[word]
    if not choices:
        return None
    words = list(choices)
    counts = [choices[w] for w in words]
    if temperature <= 0:
        return words[counts.index(max(counts))]      # greedy: the likeliest word
    weights = [c ** (1.0 / temperature) for c in counts]  # higher temp = flatter odds
    return rng.choices(words, weights=weights, k=1)[0]

def generate(start, temperature, seed, length=8):
    rng = random.Random(seed)
    out = [start]
    for _ in range(length):
        nxt = next_word(out[-1], temperature, rng)
        if nxt is None:
            break
        out.append(nxt)
    return " ".join(out)

print("temperature 0.0 (greedy — deterministic):")
print("   ", generate("the", temperature=0.0, seed=1))
print("   ", generate("the", temperature=0.0, seed=2), "  <- identical; the seed is ignored")
print("temperature 0.9 (sampled — varies):")
print("   ", generate("the", temperature=0.9, seed=1))
print("   ", generate("the", temperature=0.9, seed=2))

🧭 At **temperature 0** the model always takes the single most likely next token,
so the output is deterministic — the same every run. Turn the temperature up and
it *samples* less likely continuations: more variety, and more risk. For pulling
structured data out of text (this lesson's job), you want temperature low —
predictable beats creative.

And notice what the toy never does: check a single word against reality. It only
knows "what tends to follow what." That is the seed of the next idea.

### Context window

The model reads a bounded amount of text at once — its **context window**.
Everything has to fit inside it: your instructions, the data you paste in, the
conversation so far, *and* room for the reply. Past the edge, the earliest
tokens fall out of view.

In [ ]:
# How much of the Lesson 2 retail file could you paste into one prompt?
retail_preview = pd.read_csv("../course_data/online_retail_sample.csv")
one_row_text = retail_preview.iloc[0].to_json()
tokens_per_row = estimate_tokens(one_row_text)

for window in [200_000, 1_000_000]:      # today's Claude models: ~200K; Claude 5 family: up to 1M
    rows = window // tokens_per_row
    print(f"context {window:>9,} tokens  ~  {rows:>8,} rows of this file at once")

### Four tools, four property profiles

An LLM is not a search engine, a database, or a calculator. Each answers a
question, but each keeps a *different promise*:

| Tool | You give it | It returns | Property you can trust |
| --- | --- | --- | --- |
| Calculator | `128 × 641` | `82048` | exact, deterministic, correct |
| Database (Lesson 2) | a key | the stored row | faithful to what was stored |
| Search engine | a query | ranked existing pages | the pages really exist |
| LLM | a prompt | freshly generated text | plausible — and nothing more |

🧭 **Hallucination is not a bug to be patched — it's this table's bottom row
stated plainly.** A model emits the most *plausible* next tokens, not the *true*
ones. When plausible and true coincide, it looks like knowledge; when they
diverge, you get fluent, confident, wrong text. "Plausible next token" is simply
not the same property as "true." Everything in Unit 4 is about living with that.

> **Unit 1 takeaway:** an LLM runs one move — predict the next token — sampled at
> some temperature, over a bounded context. Useful, but its output ships with no
> guaranteed properties, truth least of all.

## Unit 2 — The API call  ·  ~45 min

Now the mechanics. An LLM endpoint is a REST API exactly like Lesson 6's: a URL,
a JSON body, headers, a JSON response. Everything new is in the *body* you send
and in what you may believe about the reply.

### The messages format

A request carries a list of **messages**, each a dict with a `role` and
`content`. Two roles matter now:

- `system` — standing instructions, set once ("You are a precise classifier. Reply with JSON only.")
- `user` — the actual request ("Classify this headline: …")

The model reads them in order and continues the conversation with an `assistant`
message — the reply.

In [ ]:
system_prompt = "You are a precise assistant. Answer in one short sentence."
messages = [
    {"role": "user", "content": "Name the capital of France."},
]
print(json.dumps({"system": system_prompt, "messages": messages}, indent=2))

🧭 **System vs user.** The system prompt is the rules of the game; the user
message is the move. The same words behave differently in each slot — a system
prompt frames everything that comes after it. Get the framing right once, reuse
it for every call.

### A first call via `requests` — Lesson 6's pattern, new endpoint

Here is the request exactly as Lesson 6 taught it: `requests.post` to a URL, with
headers and a JSON body. The only new pieces are the endpoint, two headers
(`x-api-key`, `anthropic-version`), and the body shape.

In [ ]:
import requests

ANTHROPIC_URL = "https://api.anthropic.com/v1/messages"
MODEL = "claude-haiku-4-5-20251001"   # a small, cheap model — a fine default for classification

def call_via_requests(messages, system=None, max_tokens=200):
    """The Lesson 6 pattern, pointed at the Messages endpoint. Defined for
    reference; we do not call it live in class."""
    headers = {
        "x-api-key": os.environ.get("ANTHROPIC_API_KEY", ""),
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    }
    body = {"model": MODEL, "max_tokens": max_tokens, "messages": messages}
    if system is not None:
        body["system"] = system
    response = requests.post(ANTHROPIC_URL, headers=headers, json=body, timeout=30)
    response.raise_for_status()
    return response.json()

print("Defined call_via_requests(): same shape as Lesson 6 — url, headers, json body, .json() back.")

### The SDK equivalent

The `anthropic` SDK wraps that exact HTTP call so you don't hand-build headers.
Same request, less plumbing.

In [ ]:
def call_via_sdk(messages, system=None, max_tokens=200):
    """The same call through the anthropic SDK (v0.64.0 installed). Under the
    hood it POSTs to the same URL with the same headers."""
    client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY from the environment
    kwargs = {"model": MODEL, "max_tokens": max_tokens, "messages": messages}
    if system is not None:
        kwargs["system"] = system
    return client.messages.create(**kwargs)

print("Defined call_via_sdk(): the convenience door to the same room.")

🧭 Two doors, one room. The SDK is convenience over the raw POST — like
`pd.read_csv` over parsing a file by hand in Lesson 2. Both send the identical
request; pick whichever reads better to you.

### Running offline — record once, replay forever

We have no key in class, and we don't want class to depend on a network or a
bill. So we do exactly what Lesson 2 did with pinned data files: **record** a few
responses and **replay** them. A recorded response is just the API's JSON, saved
to disk in its exact wire shape.

**Honest note:** the fixtures below are *authored* teaching examples, written by
hand in the real wire format — not captures of live calls. The token counts are
plausible estimates, not measured usage.

In [ ]:
# Each fixture has the exact shape a live call returns. The whole notebook only
# ever calls call_llm() below, so it behaves the same online or offline.
FIXTURES = {
    "capital_france": {
        "id": "msg_teaching_capital", "type": "message", "role": "assistant", "model": MODEL,
        "content": [{"type": "text", "text": "The capital of France is Paris."}],
        "stop_reason": "end_turn", "usage": {"input_tokens": 24, "output_tokens": 9},
    },
    "classify_clean": {
        "id": "msg_teaching_c1", "type": "message", "role": "assistant", "model": MODEL,
        "content": [{"type": "text", "text": "{\"category\": \"employment\", \"confidence\": 0.95}"}],
        "stop_reason": "end_turn", "usage": {"input_tokens": 60, "output_tokens": 12},
    },
    "classify_fenced": {
        "id": "msg_teaching_c2", "type": "message", "role": "assistant", "model": MODEL,
        "content": [{"type": "text", "text": "```json\n{\"category\": \"employment\", \"confidence\": 0.9}\n```"}],
        "stop_reason": "end_turn", "usage": {"input_tokens": 60, "output_tokens": 15},
    },
    "classify_prose": {
        "id": "msg_teaching_c3", "type": "message", "role": "assistant", "model": MODEL,
        "content": [{"type": "text", "text": "Sure! Here you go:\n{\"category\": \"employment\", \"confidence\": 0.88}\nHope that helps!"}],
        "stop_reason": "end_turn", "usage": {"input_tokens": 62, "output_tokens": 40},
    },
    "classify_missing_key": {
        "id": "msg_teaching_c4", "type": "message", "role": "assistant", "model": MODEL,
        "content": [{"type": "text", "text": "{\"category\": \"employment\"}"}],
        "stop_reason": "end_turn", "usage": {"input_tokens": 60, "output_tokens": 6},
    },
    "extract_acme": {
        "id": "msg_teaching_e1", "type": "message", "role": "assistant", "model": MODEL,
        "content": [{"type": "text", "text": "{\"company\": \"Acme Corp\", \"metric\": \"revenue\", \"value_pct\": 12.5}"}],
        "stop_reason": "end_turn", "usage": {"input_tokens": 44, "output_tokens": 22},
    },
    "bad_sum": {
        "id": "msg_teaching_s1", "type": "message", "role": "assistant", "model": MODEL,
        "content": [{"type": "text", "text": "6400"}],
        "stop_reason": "end_turn", "usage": {"input_tokens": 45, "output_tokens": 3},
    },
}
for name, payload in FIXTURES.items():
    (FIXTURE_DIR / f"{name}.json").write_text(json.dumps(payload, indent=2))
print("recorded", len(FIXTURES), "fixtures ->", FIXTURE_DIR)

In [ ]:
def call_llm(messages, system=None, fixture=None, max_tokens=200):
    """One entry point for the whole notebook. If a real key is set, call live
    (via the SDK); otherwise replay the named recorded fixture and say so."""
    if HAVE_KEY:
        message = call_via_sdk(messages, system=system, max_tokens=max_tokens)
        return message.model_dump() if hasattr(message, "model_dump") else message
    if fixture is None:
        raise RuntimeError("no API key set and no fixture named — nothing to replay")
    print(f"[replay] no API key -> replaying recorded fixture {fixture!r}")
    return json.loads((FIXTURE_DIR / f"{fixture}.json").read_text())

def text_of(response):
    """The model's generated text lives at content[0].text in the wire shape."""
    return response["content"][0]["text"]

response = call_llm(
    [{"role": "user", "content": "Name the capital of France."}],
    system="Answer in one short sentence.",
    fixture="capital_france",
)
print("model :", response["model"])
print("reply :", text_of(response))
print("usage :", response["usage"])

🧭 The reply carries a `usage` block — the **authoritative** token counts, input
and output. That's what you are billed on, and what our character-count estimate
was only guessing at.

### Tokens are the cost unit — price a real, invoice-sized job

You pay per token: input tokens (what you send) and output tokens (what comes
back), at different rates. Let's price a concrete job — classify every line of
the Lesson 2 retail extract.

**Pricing honesty:** the numbers below were fetched from the Anthropic pricing
page on **2026-07-11**. Prices change; treat these as a dated snapshot and check
the live page (see Extensions) before quoting a cost. Model ids in this course:
`claude-haiku-4-5-20251001`, `claude-sonnet-5`, `claude-opus-4-8` (newest family:
Claude 5).

In [ ]:
retail = pd.read_csv("../course_data/online_retail_sample.csv")
print("invoice lines:", len(retail))

# Send each product description with a short instruction; get a short JSON label.
PROMPT_OVERHEAD = 40          # standing instruction, per call (estimate)
OUTPUT_TOKENS_PER_CALL = 15   # a short JSON label (estimate)

input_tokens = sum(PROMPT_OVERHEAD + estimate_tokens(str(d)) for d in retail["Description"])
output_tokens = OUTPUT_TOKENS_PER_CALL * len(retail)
print("estimated input tokens :", input_tokens)
print("estimated output tokens:", output_tokens)

In [ ]:
# US dollars per 1,000,000 tokens, as (input_rate, output_rate).
# Fetched 2026-07-11 — a dated snapshot, not a live quote.
PRICES_USD_PER_MTOK = {
    "claude-haiku-4-5-20251001": (1.0, 5.0),
    "claude-sonnet-5":           (2.0, 10.0),   # introductory rate, through 2026-08-31
    "claude-opus-4-8":           (5.0, 25.0),
}

def cost(input_tokens, output_tokens, model):
    in_rate, out_rate = PRICES_USD_PER_MTOK[model]
    return (input_tokens * in_rate + output_tokens * out_rate) / 1_000_000

for model in PRICES_USD_PER_MTOK:
    print(f"{model:28s} ${cost(input_tokens, output_tokens, model):.6f}  for all {len(retail)} lines")

🧭 **Two levers, one bill.** Same job, a ~5x cost swing from model choice alone —
so "the cheapest model that can do the task" is a real decision, not a detail.
And cost is *linear* in tokens (Lesson 4's O(n)): 60 lines cost pennies, 60
million cost the same per line — so you estimate on a sample and multiply, which
is exactly what Unit 4 does before spending anything.

> **Unit 2 takeaway:** the call is Lesson 6's REST request with a messages body;
> the SDK is a convenience wrapper over it; you pay per input + output token; and
> you can run and test the whole thing offline by replaying recorded responses.

---
## ⏱️ Session 2 warm-up  ·  5 questions from last time

Answer from memory first, then check.

1. What is the single mechanical move an LLM makes, over and over?
2. What does temperature 0 do to the output, and why does that suit classification?
3. A hallucination is the direct consequence of two properties not being the same. Which two?
4. In the messages format, what is the difference between a `system` and a `user` message?
5. You pay per token. Name the two token counts you're billed on, and where the *authoritative* numbers come from.

<details><summary><b>Answers</b></summary>

1. Predict the next token from the text so far, append it, and repeat.
2. It makes the model pick the single most likely next token every time, so the
   output is deterministic — predictable beats creative when you're extracting data.
3. "Plausible next token" and "true". The model optimizes the first; nothing
   guarantees the second.
4. `system` sets standing instructions (the rules), set once; `user` carries the
   actual request (the move). The system prompt frames everything after it.
5. Input tokens and output tokens. The authoritative counts come from the
   response's `usage` block (or the model's tokenizer) — not a character estimate.
</details>

## Unit 3 — Structured output  ·  ~50 min

For a *program* to use a model's answer, the answer needs a shape a program can
parse — usually **JSON**. So we ask for JSON. The whole point of this unit is the
catch: asking for JSON gets you JSON *usually*, and "usually" is not a property
you can build on. So we validate every reply and retry the ones that fail —
Lesson 5's *validate-early* habit, now aimed at a source that can hand you
anything.

### Asking for JSON

The instruction goes in the system prompt. We reuse the exercise's six economic
categories.

In [ ]:
CLASSIFY_SYSTEM = (
    "You are an economic-news classifier. Reply with ONLY a JSON object of the form "
    '{"category": <one of: monetary_policy, inflation, employment, markets, trade, housing>, '
    '"confidence": <number 0-1>}. No prose, no code fence.'
)
headline = "Unemployment falls to 3.9% as hiring accelerates"
response = call_llm([{"role": "user", "content": headline}],
                    system=CLASSIFY_SYSTEM, fixture="classify_clean")
print(text_of(response))

### Watching it come back malformed

You asked nicely. Real models mostly comply — and sometimes don't. Here are three
replies to the same kind of request, each a way JSON arrives broken in practice:
a code-fence wrapper, prose wrapped around the JSON, and a missing required key.

In [ ]:
for fixture in ["classify_fenced", "classify_prose", "classify_missing_key"]:
    reply = text_of(call_llm([{"role": "user", "content": headline}],
                             system=CLASSIFY_SYSTEM, fixture=fixture))
    print(f"--- {fixture} ---")
    print(repr(reply))
    print()

### Validate with `json.loads` plus property checks

Two gates, every reply. **Gate 1:** does it even parse? `json.loads` turns text
into a Python object or raises. **Gate 2:** does the parsed object have the
properties we require — the right keys, a category from the legal set, a
confidence that is a real number in `0..1`? This is Lesson 1's property checklist
written as code.

In [ ]:
def parse_json(text):
    """Gate 1 — is it JSON at all? Return the object, or None."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

VALID_CATEGORIES = {"monetary_policy", "inflation", "employment", "markets", "trade", "housing"}

def is_valid(payload):
    """Gate 2 — does the parsed object satisfy the schema?"""
    if not isinstance(payload, dict):
        return False
    if set(payload) != {"category", "confidence"}:
        return False
    if payload["category"] not in VALID_CATEGORIES:
        return False
    confidence = payload["confidence"]
    if isinstance(confidence, bool) or not isinstance(confidence, (int, float)):
        return False
    return 0.0 <= confidence <= 1.0

for fixture in ["classify_clean", "classify_fenced", "classify_prose", "classify_missing_key"]:
    text = text_of(call_llm([{"role": "user", "content": headline}],
                            system=CLASSIFY_SYSTEM, fixture=fixture))
    payload = parse_json(text)
    parses = payload is not None
    print(f"{fixture:22s} parses={str(parses):5s} valid={is_valid(payload) if parses else False}")

🧭 The clean reply passes both gates. The fenced and prose replies fail Gate 1 —
they aren't JSON, the ` ``` ` and the "Sure!" break `json.loads`. The missing-key
reply passes Gate 1 and fails Gate 2. An LLM reply has *no* property until you
check it — not even "is JSON". Same discipline as Lesson 5's validate-at-the-
boundary; the boundary just leaks in more ways now.

### Retry-on-invalid as a loop

When a reply fails, the fix is usually to ask again — the model is stochastic
(remember temperature), so the next draw is often clean. A **bounded** retry loop
tries a few times, then gives up rather than looping forever. Below, the "model"
is a recorded *sequence*: attempt 0 is malformed, attempt 1 is clean, so the loop
recovers on the retry — deterministically, because the fixtures are pinned.

In [ ]:
RETRY_SEQUENCE = ["classify_fenced", "classify_clean"]   # attempt 0 bad, attempt 1 good
MAX_ATTEMPTS = 3

def classify_with_retry(headline, sequence):
    for attempt in range(MAX_ATTEMPTS):
        fixture = sequence[min(attempt, len(sequence) - 1)]
        text = text_of(call_llm([{"role": "user", "content": headline}],
                                system=CLASSIFY_SYSTEM, fixture=fixture))
        payload = parse_json(text)
        if payload is not None and is_valid(payload):
            return {"payload": payload, "retries": attempt}
        print(f"  attempt {attempt}: invalid ({fixture}) -> retrying")
    return {"payload": None, "retries": MAX_ATTEMPTS - 1}

result = classify_with_retry(headline, RETRY_SEQUENCE)
print("result:", result)

🧭 The loop turned an unreliable source into a reliable one — not by trusting
harder, but by checking and retrying. The retry *count* is data worth keeping: a
high retry rate is a signal your prompt needs work (or your model is too small).
Measuring exactly that is the heart of this lesson's exercise.

### Classification and extraction — the two workhorses

Almost every practical LLM data task is one of two shapes:

- **Classification** — map free text to one of a fixed set of labels (what we just
  did; the property to enforce is *set membership*).
- **Extraction** — pull structured fields out of messy text (the property to
  enforce is *schema shape and field types*).

Both end the same way: a JSON object you validate before you trust.

In [ ]:
EXTRACT_SYSTEM = (
    "Extract the fields from the sentence. Reply with ONLY JSON: "
    '{"company": <string>, "metric": <string>, "value_pct": <number>}.'
)
sentence = "Acme Corp said quarterly revenue rose 12.5% from a year earlier."
payload = parse_json(text_of(call_llm([{"role": "user", "content": sentence}],
                                      system=EXTRACT_SYSTEM, fixture="extract_acme")))
print("extracted:", payload)

def extraction_is_valid(p):
    return (isinstance(p, dict)
            and set(p) == {"company", "metric", "value_pct"}
            and isinstance(p["company"], str)
            and isinstance(p["metric"], str)
            and isinstance(p["value_pct"], (int, float)) and not isinstance(p["value_pct"], bool))

print("valid extraction:", extraction_is_valid(payload))

> **Unit 3 takeaway:** ask for JSON, but trust nothing until it passes two gates —
> `json.loads` (is it JSON?) and a property check (is it the *right* JSON?). Retry
> the failures in a bounded loop, and count how often you had to.

## Unit 4 — Trust boundaries  ·  ~30 min

The model works. The last unit is the discipline that keeps it from hurting you —
three boundaries: what you **send**, what you **believe**, and what you **spend**.

### What to send — privacy

Your prompt leaves your machine and is processed by a third party. Treat it like
anything you'd put in an email to a vendor: no secrets, no credentials, no
personal data you aren't cleared to share. And **minimize** — send the smallest
slice of data that lets the model do the job.

In [ ]:
raw = {"customer_id": "00101", "email": "alice@example.com",
       "note": "asking about a refund on order T007"}

# The task only needs the free-text note — so send only that.
to_send = {"note": raw["note"]}
print("on your machine :", raw)
print("sent to the API :", to_send)

🧭 The model can't leak what it never received. This is the privacy version of
Lesson 2's "load only the columns the question needs". (And a key belongs in an
environment variable, never pasted into a prompt or hard-coded in a notebook —
which is why `call_llm` reads it from the environment and never prints it.)

### What to believe — verify numbers in Python, never in the model

A model predicts plausible tokens. "The sum of these numbers" is a
plausible-*looking* number, and often a wrong one. Never let the model do
arithmetic — or counting, or sorting, or lookups — you can do exactly in Python.
Let Python compute; let the model phrase.

In [ ]:
numbers = [1200, 950, 1400, 1800, 1250]
question = f"What is the sum of {numbers}? Reply with just the number."

model_answer = text_of(call_llm([{"role": "user", "content": question}], fixture="bad_sum"))
python_answer = sum(numbers)

print("model says :", model_answer)
print("python says:", python_answer)
print("agree?     :", str(python_answer) in model_answer)

🧭 Fluent, confident, and off by 200. If a number matters, compute it. This is
Lesson 4's closing rule, verbatim: *let Python run the exact algorithms — sums,
sorts, matches — and let the model interpret the verified results, never the
reverse.* The model's real job here is language: turning your verified `6600`
into "Sales totaled $6,600 across the five months."

### What to spend — cost control

Three habits keep the bill sane: pick the smallest model that does the job
(Unit 2's table), send fewer tokens (short prompts, minimal data), and cap
`max_tokens` so a runaway reply can't run up the cost. And **estimate on a sample,
then multiply** — a forecast you can make before spending a cent.

In [ ]:
sample_cost = cost(input_tokens, output_tokens, "claude-haiku-4-5-20251001")
print("cheapest model, per scale (Haiku 4.5, dated snapshot):")
for scale in [1, 1_000, 100_000]:
    print(f"  {scale * len(retail):>10,} lines  ~  ${sample_cost * scale:>10.2f}")

🧭 Same "measure on small, reason about big" move as Lesson 4 — except here it
maps straight to dollars.

### Recorded fixtures — why the whole notebook ran offline

Every call above replayed a recorded response (`call_llm(..., fixture=...)`).
That isn't just a class convenience: replaying pinned responses is how you **test**
LLM code — deterministic inputs, so you can check your validation and retry logic
without paying, waiting, or depending on the network. The exercise is built
entirely this way.

> **Unit 4 takeaway:** send the least data that does the job; verify every number
> in Python; forecast cost before you spend; and pin responses so your LLM code is
> testable and offline-runnable.

## Wrap-up

### What you can do now

- explain tokens, next-token prediction, context windows, and temperature
- make the same API call two ways — raw `requests` and the SDK — online or replayed offline
- price a job in tokens and choose a model on cost
- ask for JSON, validate every reply against a schema, and retry the failures in a bounded loop
- hold the three trust boundaries: send little, believe nothing you can check in Python, cap the spend

### The lesson, one sentence

> *An LLM endpoint is just another API — so treat its reply as data with no
> properties yet: give it a shape by asking for JSON, give it truth by validating
> and by verifying numbers in Python, and give it a budget in tokens.*

### Where each earlier idea reappears

- **Lesson 1** (properties) → an LLM reply arrives with none; you assign every one
- **Lesson 2** (validate at load) → validate at the boundary, now with more failure modes
- **Lesson 4** (O(n), "let Python run the algorithms") → token cost is linear; never let the model do the arithmetic
- **Lesson 5** (validate early, small functions) → `parse_json` + `is_valid` + a bounded retry loop
- **Lesson 6** (the REST call) → the identical request shape, a very different reply

### Next

Open **`../lesson11_exercise/`** and follow its `README.md`: classify 20 economic-news
headlines into the six categories, validate every reply against the schema, retry
the malformed ones, and report how many retries the run needed — all offline,
against pinned fixtures. The checker pins the schema, the retry counts, and the
run totals.

### Extensions

- [Anthropic API quickstart](https://docs.claude.com/en/docs/get-started) — your first real call, key setup included
- [Anthropic pricing](https://docs.claude.com/en/docs/about-claude/pricing) — the live, current token prices (the snapshot in Unit 2 will drift)
- [The Tokenizer Playground](https://huggingface.co/spaces/Xenova/the-tokenizer-playground) — watch text split into tokens
- [Understanding JSON Schema](https://json-schema.org/understanding-json-schema/) — the grammar for the property checks in Unit 3
- [Messages API reference](https://docs.claude.com/en/api/messages) — every field of the request and response